# DS-A1 | Data Provenance and Measurement Audit

## Research question

Does the timing of crossing 52°N during southward migration differ between tracked juvenile and adult Black-tailed Godwits (*Limosa limosa limosa*)?

## Dataset

This analysis uses the public dataset associated with Verhoeven et al. (2022), “Age-dependent timing and routes demonstrate developmental plasticity in a long-distance migratory bird.”

The analysis is restricted to southward migration timing at the 52°N latitude boundary, which represents departure from the breeding area in the Netherlands.

## 1. Data source, license, version, and file identity

**Dataset:** *Age-dependent timing and routes demonstrate developmental plasticity in a long-distance migratory bird*

**Authors:** Mo A. Verhoeven, A. H. Jelle Loonstra, Alice D. McBride, Wiebe Kaspersma, Jos C. E. W. Hooijmeijer, Christiaan Both, Nathan R. Senner, and Theunis Piersma

**Repository:** Dryad Digital Repository

**DOI:** 10.5061/dryad.02v6wwq4j

**Associated article:** Verhoeven et al. (2022), *Journal of Animal Ecology*, 91: 566–579

**File used:** `2.Adult_and_juvenile_timing_forR.csv`

**License:** CC0

In [3]:
import hashlib

file_path = "2.Adult_and_juvenile_timing_forR.csv"

sha256 = hashlib.sha256()

with open(file_path, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        sha256.update(chunk)

print("SHA-256:", sha256.hexdigest())

SHA-256: ef3b0967eac749481aaf7f9220ae617ff70daed3d8f63296dc326ca34947df59


## 2. Analytical framework

The analysis is restricted to southward migration timing at the 52°N latitude boundary and compares juvenile and adult Black-tailed Godwits tracked during the same migration years.

**Stakeholder:** Researchers studying age-related differences and the development of migratory behaviour in Black-tailed Godwits.

**Population:** Continental Black-tailed Godwits (*Limosa limosa limosa*) breeding in the study area in the Netherlands.

**Sample:** Tracked juvenile and adult Black-tailed Godwits with usable southward migration timing records at 52°N during 2016 and 2017.

**Unit of analysis:** One southward migration event of one tracked individual in one year.

**Target variable:** Day of year on which the bird crossed 52°N during southward migration.

**Estimand:** The difference in mean day-of-year of 52°N crossing between juvenile and adult southward migration records.

## 3. Data-generating process and provenance lineage

### Data-generating process

For juvenile godwits:

Juvenile bird  
→ satellite PTT transmitter  
→ Argos location records  
→ filtered migration track  
→ estimated crossing of the 52°N latitude boundary  
→ crossing date recorded in the dataset

For adult godwits:

Adult bird  
→ light-level geolocator  
→ recorded light-level data  
→ migration schedule reconstructed with FLightR  
→ estimated crossing of the 52°N latitude boundary  
→ crossing date recorded in the dataset

The same biological quantity — timing of crossing 52°N during southward migration — was therefore obtained using different measurement systems for juveniles and adults.

### Provenance lineage

Field measurements and tracking data  
→ processing and reconstruction by the study authors  
→ published Dryad dataset  
→ `2.Adult_and_juvenile_timing_forR.csv`  
→ local copy used in this notebook  
→ filtered analytical dataset for southward migration at 52°N in 2016–2017

## 4. Data quality audit

The dataset was audited for schema, value ranges, missingness, duplicates, anomalies, and potential leakage. Checks were performed on the full source dataset and, where relevant, on the analytical subset used for the research question.

In [4]:
import pandas as pd

file_path = "2.Adult_and_juvenile_timing_forR.csv"

df = pd.read_csv(file_path)

analysis_df = df[
    (df["What 2"] == "Southward Migration") &
    (df["Latitude"] == 52) &
    (df["To Breed"].isin([2016, 2017]))
].copy()

print("Full dataset shape:", df.shape)
print("Analytical subset shape:", analysis_df.shape)
print("\nAge groups in analytical subset:")
print(analysis_df["Age"].value_counts())

Full dataset shape: (1670, 10)
Analytical subset shape: (78, 10)

Age groups in analytical subset:
Age
Ad     50
Juv    28
Name: count, dtype: int64


### 4.1 Schema check

The schema check verifies the structure of the dataset, including the number of rows and columns, variable names, and data types.

In [5]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFull dataset dimensions:")
print(df.shape)

print("\nAnalytical subset dimensions:")
print(analysis_df.shape)

Column names:
['Individual', 'Individual2', 'Q50', 'dQ50', 'Date', 'Latitude', 'To Breed', 'Sahara', 'Age', 'What 2']

Data types:
Individual      object
Individual2     object
Q50             object
dQ50            object
Date           float64
Latitude         int64
To Breed         int64
Sahara          object
Age             object
What 2          object
dtype: object

Full dataset dimensions:
(1670, 10)

Analytical subset dimensions:
(78, 10)


The dataset contains 1,670 rows and 10 variables. The analytical subset contains 78 rows and the same 10 variables.

Most identifier and categorical variables were read as text (`object`). `Latitude` and `To Breed` were read as integers, while `Date`, which is used as the numerical representation of migration timing, was read as a floating-point variable.

The date-related variables `Q50` and `dQ50` were stored as text in the source file, but this does not affect the present analysis because migration timing is represented numerically by `Date`.

### 4.2 Range check

The range check examines whether the values of key variables fall within plausible and expected ranges for the present analysis.

In [6]:
print("Unique age classes:")
print(analysis_df["Age"].unique())

print("\nYears represented:")
print(sorted(analysis_df["To Breed"].unique()))

print("\nLatitude values represented:")
print(sorted(analysis_df["Latitude"].unique()))

print("\nMigration direction values represented:")
print(analysis_df["What 2"].unique())

print("\nDate summary:")
print(analysis_df["Date"].describe())

Unique age classes:
['Juv' 'Ad']

Years represented:
[np.int64(2016), np.int64(2017)]

Latitude values represented:
[np.int64(52)]

Migration direction values represented:
['Southward Migration']

Date summary:
count     78.000000
mean     194.320513
std       25.978259
min      153.000000
25%      177.000000
50%      185.500000
75%      218.000000
max      257.000000
Name: Date, dtype: float64


All categorical variables in the analytical subset contained only the expected values: two age classes (`Juv` and `Ad`), years 2016 and 2017, latitude 52°N, and southward migration records only.

The numerical migration-timing variable `Date` ranged from day 153 to day 257 of the year. These values fall within a plausible seasonal range for southward migration and no impossible day-of-year values were detected.

### 4.3 Missingness check

The missingness check examines whether any variables contain missing values and whether missing data affect the variables required for the research question.

In [7]:
print("Missing values in the full dataset:")
print(df.isna().sum())

print("\nMissing values in the analytical subset:")
print(analysis_df.isna().sum())

Missing values in the full dataset:
Individual        0
Individual2       0
Q50               1
dQ50              1
Date              1
Latitude          0
To Breed          0
Sahara         1361
Age               0
What 2            0
dtype: int64

Missing values in the analytical subset:
Individual      0
Individual2     0
Q50             0
dQ50            0
Date            0
Latitude        0
To Breed        0
Sahara         50
Age             0
What 2          0
dtype: int64


The full dataset contains one missing value in each of `Q50`, `dQ50`, and `Date`, and a large number of missing values in `Sahara`.

However, the analytical subset contains no missing values in the variables required for the research question, including `Age`, `Date`, `Latitude`, `To Breed`, `What 2`, and the individual identifiers.

The `Sahara` variable contains 50 missing values in the analytical subset, but this variable is not used in the present analysis. Therefore, missingness does not affect the defined estimand.

### 4.4 Duplicate check

The duplicate check examines whether the analytical subset contains repeated rows or repeated migration records that could be mistaken for duplicates.

In [8]:
print("Fully duplicated rows in the full dataset:")
print(df.duplicated().sum())

print("\nFully duplicated rows in the analytical subset:")
print(analysis_df.duplicated().sum())

print("\nDuplicated Individual2 values in the analytical subset:")
print(analysis_df["Individual2"].duplicated().sum())

print("\nUnique individuals in the analytical subset:")
print(analysis_df["Individual"].nunique())

print("\nNumber of analytical records:")
print(len(analysis_df))

Fully duplicated rows in the full dataset:
0

Fully duplicated rows in the analytical subset:
0

Duplicated Individual2 values in the analytical subset:
0

Unique individuals in the analytical subset:
71

Number of analytical records:
78


No fully duplicated rows were detected in either the full dataset or the analytical subset. The `Individual2` identifier was also unique for all analytical records.

The analytical subset contains 78 migration records from 71 unique individuals. Therefore, seven records represent repeated observations of the same individuals in different years. These repeated observations are not data duplicates, but they mean that not all analytical records are statistically independent at the individual level.

### 4.5 Anomaly check

The anomaly check examines the analytical subset for unexpected or implausible values that could indicate data-entry, coding, or processing problems.

In [9]:
print("Unique values in Age:")
print(analysis_df["Age"].unique())

print("\nUnique values in Latitude:")
print(analysis_df["Latitude"].unique())

print("\nUnique values in To Breed:")
print(analysis_df["To Breed"].unique())

print("\nUnique values in What 2:")
print(analysis_df["What 2"].unique())

print("\nDate values outside the valid day-of-year range:")
print(analysis_df[(analysis_df["Date"] < 1) | (analysis_df["Date"] > 366)])

print("\nRecords with missing identifiers:")
print(analysis_df[
    analysis_df["Individual"].isna() |
    analysis_df["Individual2"].isna()
])

Unique values in Age:
['Juv' 'Ad']

Unique values in Latitude:
[52]

Unique values in To Breed:
[2016 2017]

Unique values in What 2:
['Southward Migration']

Date values outside the valid day-of-year range:
Empty DataFrame
Columns: [Individual, Individual2, Q50, dQ50, Date, Latitude, To Breed, Sahara, Age, What 2]
Index: []

Records with missing identifiers:
Empty DataFrame
Columns: [Individual, Individual2, Q50, dQ50, Date, Latitude, To Breed, Sahara, Age, What 2]
Index: []


No unexpected categorical values were detected in the analytical subset. All records were classified as either juvenile or adult, belonged to 2016–2017, represented the 52°N latitude boundary, and corresponded to southward migration.

No `Date` values fell outside the valid day-of-year range of 1–366, and no records had missing individual identifiers. No obvious coding or range anomalies were detected.

### 4.6 Leakage check

No target leakage is present in the defined analysis because `Age` is used only as the grouping variable and `Date` as the outcome.

Several variables require caution in a predictive setting. `Q50`, `dQ50`, and `Date` encode the same 52°N crossing event in different formats, so using `Q50` or `dQ50` to predict `Date` would constitute direct target leakage.

Variables describing later migration outcomes, such as `Sahara`, could also introduce temporal leakage if they were used to predict an earlier event such as departure timing. Individual identifiers may additionally encode repeated individuals or year-specific information and should not be used as predictive features without careful justification.

None of these variables are used as predictors in the present analysis.

## 5. Bias, measurement counterexample, and prohibited claim

**Selection bias:** The final timing sample is not a random subset of all tagged birds. Of 80 tagged juveniles, usable migration tracks were obtained for only 28, largely because many juveniles died before southward migration. Adult geolocator data also depended on successful recapture and device retrieval. Therefore, birds with usable timing records may differ systematically from all tagged individuals.

**Measurement counterexample:** Juvenile and adult migration timing was measured using different tracking systems. Juvenile timing was derived from satellite PTT locations with a limited duty cycle, whereas adult timing was reconstructed from light-level geolocator data using FLightR. If the two methods introduce different systematic errors in estimating the date of crossing 52°N, part of the observed age-group difference could reflect the measurement method instead of a true biological difference.

**Additional limitation:** Data from 2016 and 2017 were pooled when comparing age groups. If migration timing differed between years because of environmental or other interannual effects, pooling the two years could influence the apparent difference between juveniles and adults.

**Prohibited claims:** The dataset can show that juvenile and adult godwits differed in measured departure timing, but it cannot explain why this difference occurred. Claims about causal mechanisms, such as developmental plasticity, inherited migration programs, or experience, would therefore go beyond the evidence provided by these timing data. The observed pattern should also not be assumed to apply to other populations, years, or environmental conditions.